In [ ]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset
from massspecgym.data.transforms import InMemCachedMolTransform
from chemembed_transforms import (
    ChemEmbedSpecTransform,
    molformer_factory
)
from ann_model_massspecgym import AnnRetrieval
from pytorch_lightning import Trainer
from pathlib import Path
import pandas as pd

# 1) Embedding

In [ ]:
# MoLFormer (768 dims)
mol_transform_factory = molformer_factory
embedding_name = "molformer"
mol_embedding_dim = 768

selected_molecular_embedding = InMemCachedMolTransform(
    # cache_pth=Path(f"cache.pkl"),
    mol_transform_factory=mol_transform_factory,
    verbose=True
)

# 2) Load model

In [ ]:
model_name = "ANN_trained_molformer.ckpt"
model = AnnRetrieval.load_from_checkpoint(
    f"models/{model_name}",
    mol_embedding_dim=mol_embedding_dim
)
model.eval()

# 3) Treat dataset

In [ ]:
test_dataset = RetrievalDataset(
    candidates_pth="standard",
    spec_transform=ChemEmbedSpecTransform(),
    # mol_transform=selected_molecular_embedding,
)

data_module_test = MassSpecDataModule(
    dataset=test_dataset,
    batch_size=32,
    num_workers=7
)

data_module_test.setup(stage="test")

# 4) Evaluate model

In [ ]:
trainer = Trainer(
    accelerator="auto",
    log_every_n_steps=1,
    enable_progress_bar=True,
)

results = trainer.test(model, datamodule=data_module_test)

print(results)

pd.DataFrame(results).to_csv("results.csv", index=False)